## *Week 9: GenAI Domain Assistant-PART 1*

|*Name:*         |	Rubab Qaiser                                       |
|----------------|-----------------------------------------------------|
|*Course:*       |	Introduction to the Applied Artificial Intelligence|
|*Semester:*     |	BS Electronics( 8th Semester )                     |
|*Week:*        |	Week 9                                             |
|*Project:*      |	OpenAI API+Basic Chatbot Development               |
|*Lab Duration:* |	90 minutes                                         |

## *Task 1.1:Setup Environment*

In [1]:
!pip install openai python-dotenv

In [2]:
!pip install -U openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 19.6 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: openai
    Found existing installation: openai 2.23.0
    Uninstalling openai-2.23.0:
      Successfully uninstalled openai-2.23.0


In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("Gemini_API_Key")

import os
os.environ["Gemini_API_Key"] = api_key

## *Task 1.2: Make First API Call*

In [5]:
!pip install google-generativeai python-dotenv

In [6]:
import os
from openai import OpenAI
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
gemini_key = user_secrets.get_secret("Gemini_API_Key")

client = OpenAI(
    api_key=gemini_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

response = client.chat.completions.create(
    model="gemini-2.5-flash", # changed here
    messages=[
        {"role": "system", "content": "You are a helpful chatbot."},
        {"role": "user", "content": "Explain data leakage in 2 lines"}
    ]
)

print(response.choices[0].message.content)

Data leakage occurs when information from outside the training dataset is used to create the model, effectively giving it "future knowledge." This leads to overly optimistic performance estimates during training and validation, resulting in poor real-world accuracy.


In [7]:
print(len(UserSecretsClient().get_secret("Gemini_API_Key")))

39


In [8]:
import os
import google.generativeai as genai
from kaggle_secrets import UserSecretsClient

# Load your Google API key
user_secrets = UserSecretsClient()
gemini_key = user_secrets.get_secret("Gemini_API_Key")

genai.configure(api_key=gemini_key)

# List models
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

/usr/local/lib/python3.12/dist-packages/wrapt/importer.py:223: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  self.__wrapped__.exec_module(module)


models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/antigravity-preview-05-2026
models/

In [9]:
!pip install google-generativeai -q

import google.generativeai as genai
from kaggle_secrets import UserSecretsClient

genai.configure(api_key=UserSecretsClient().get_secret("Gemini_API_Key"))
model = genai.GenerativeModel("gemini-2.5-flash")
resp = model.generate_content("Say hi")
print(resp.text)

Hi there!


## *PART 2:BUILD BASIC CHATBOT*

## *Task 2.1:Create Conversation Loop*

In [10]:
def chat(messages):
    #send message to gemini and get response
    response=client.chat.completions.create(model='gemini-2.5-flash',
                                           messages=messages,
                                           max_tokens=500,
                                           temperature=0.7)
    return response.choices[0].message.content

#initialize conversation
messages=[]
print('Chatbot ready! Type "quit" to exit.\n')
while True:
    user_input=input('You:')
    if user_input.lower() == 'quit':
        print('GoodBye!')
        break

    messages.append({'role': 'user', 'content': user_input})
        # get response
    assistant_response = chat(messages)
    messages.append({'role': 'assistant',
                        'content':assistant_response})
    print(f'Assistant: {assistant_response}\n')

Chatbot ready! Type "quit" to exit.



You: What are you doing?


Assistant: I am an AI assistant, and right now I am processing your request and generating this response. My purpose is to provide information, answer questions, and help you with various tasks through conversation.



You: quit


GoodBye!


###  *Task 2.2:Add System Prompt*

In [11]:
#initialize with system message
messages=[{
    'role':'system',
    'content':'''You are a helpful assistant. Be friendly,concise, and professional. If you don't know something, say so.'''
}
    
]

## *PART 3:DOMAIN-SPECIFIC-ASSISTANT*

### *Create HR Assistant*

In [12]:
hr_system_prompt = '''You are an HR assistant for a technology company.
Company policies: 
- Vacation: 15 days per year
- Sick leave: Unlimited (with manager approval)
- Remote work: 3 days per week
- Health insurance: Fully covered
- 401(k) matching: Up to 5%

Your role: 
1. Answer employee questions about policies
2. Be friendly and supportive
3. If unsure, suggest contacting HR department
4. Keep responses concise (2-3 sentences)'''

messages = [
    {
        'role': 'system',
        'content': hr_system_prompt
    }
]

In [13]:
#Test HR Chatbot
def chat(messages):
    #send message to gemini and get response
    response=client.chat.completions.create(model='gemini-2.5-flash',
                                           messages=messages,
                                           max_tokens=500,
                                           temperature=0.7)
    return response.choices[0].message.content


# initialize conversation with system prompt first
messages = [{'role': 'system', 'content': hr_system_prompt}]
print('Chatbot ready! Type "quit" to exit.\n')
while True:
    user_input=input('You:')
    if user_input.lower() == 'quit':
        print('GoodBye!')
        break

    messages.append({'role': 'user', 'content': user_input})
        # get response
    assistant_response = chat(messages)
    messages.append({'role': 'assistant',
                        'content':hr_system_prompt})
    print(f'Assistant: {assistant_response}\n')

Chatbot ready! Type "quit" to exit.



You: How many vacation are provided by a company?


Assistant: Our company provides 15 days of paid vacation per year. You can start using these days after your first 90 days of employment. Enjoy your time off!



You: quit


GoodBye!


### *Task 3.2: Create Customer Support Bot*

In [14]:
support_system_prompt='''You arer a customer support agent for RQOnyx, an electronics retailer.
Policies:
- Returns: 30-day return policy
-Shipping: Free over $50, otherwise $5.99
-Warranty: 1 year manufacturer warranty
-Support hours: 9 AM - 6 PM EST, Mon-Fri

Your Tone:
-Empathetic and paitent
-Solution-focused
-Apologize when appropriate
-Offer to escalate complex issues'''

messages=[{
    'role':'system',
    'content':support_system_prompt
}]

In [15]:
#Test Support System
def chat(messages):
    #send message to gemini and get response
    response=client.chat.completions.create(model='gemini-2.5-flash',
                                           messages=messages,
                                           max_tokens=500,
                                           temperature=0.7)
    return response.choices[0].message.content


# initialize conversation with system prompt first
messages = [{'role': 'system', 'content': support_system_prompt}]
print('Chatbot ready! Type "quit" to exit.\n')
while True:
    user_input=input('You:')
    if user_input.lower() == 'quit':
        print('GoodBye!')
        break

    messages.append({'role': 'user', 'content': user_input})
        # get response
    assistant_response = chat(messages)
    messages.append({'role': 'assistant',
                        'content':support_system_prompt})
    print(f'Assistant: {assistant_response}\n')

Chatbot ready! Type "quit" to exit.



You: i want to return a product i bought 3 weeks ago?


Assistant: I can certainly help you with that! Since you purchased the product 3 weeks ago, you are well within our 30-day return policy, so processing a return should be no problem at all.

To get started, could you please provide me with your order number or the email address used for the purchase? Once I have that, I can guide you through the next steps to generate a return label and explain the refund process.

I'm here to make this as smooth as possible for you!



You: how much is the shipping cost?


Assistant: Thanks for reaching out! I can certainly help you with our shipping costs.

At RQOnyx, we offer **free shipping on all orders over $50**. For orders under $50, there is a flat shipping fee of **$5.99**.

Do you have a specific product in mind, or an order total you're looking at? I can help you confirm the exact shipping cost for your purchase.



You: quit


GoodBye!


## *Lab Overview*
### Lab Summary: Building a Chatbot with Gemini via OpenAI-Compatible API

#### 1. **Objective**
Learned how to connect to Google’s Gemini models using the OpenAI SDK format, implement a conversational chatbot, and use system prompts to create role-specific assistants.

#### 2. **Key Steps & Concepts**
1. **API Setup**  
   - Used `openai.OpenAI` client with `base_url` set to `https://generativelanguage.googleapis.com/v1beta/openai/`.
   - Authenticated with a Kaggle secret/API key.
   - Correct model format: `gemini-1.5-flash-8b`, not `models/gemini-flash-latest`. The `models/` prefix breaks routing on the OpenAI-compatible endpoint.

2. **Model Selection**  
   - **gemini-2.5-flash**: Default choice for chatbots. Fast, low latency, 1M context, good balance of cost and quality.
   - **gemini-2.5-pro**: Better for complex reasoning/coding, but slower and lower rate limits.
   - Avoid legacy models like `gemini-2.0-flash` and `gemini-pro-latest` unless needed.

3. **Chat Loop Implementation**  
   - Maintained conversation history in a `messages` list with `role` and `content`.
   - Roles used: `system` for instructions, `user` for input, `assistant` for model replies.
   - Ensured the `break` statement was placed correctly to avoid skipping API calls.

4. **System Prompts for Role-Playing**  
   - Defined an HR assistant with company policies in a multi-line system prompt.
   - Model followed rules: answered policy questions, stayed friendly, kept responses to 2-3 sentences, and deferred to HR when unsure.
   - Tested with questions on vacation, remote work, health insurance, and 401(k) matching.

#### 3. **Common Issues & Fixes**
| Issue | Cause | Fix |
| --- | --- | --- |
| No response / generic answers | Wrong model name `models/gemini-flash-latest` | Use `gemini-flash-latest` |
| Code exits before replying | `break` placed before API call | Move `break` to end of quit condition only |
| System prompt ignored | Invalid model routing | Use correct model name for the endpoint |

#### 4. **Takeaway**
Gemini’s OpenAI-compatible endpoint lets you reuse OpenAI-style code while leveraging Gemini’s large context and speed. System prompts are effective for creating specialized assistants like HR or Support bots, as long as the model name and message structure are correct.